# **ginger**

In [1]:
import sys
import os

try:
    %load_ext autoreload
    %autoreload 2
    print("Autoreload enabled.")
except Exception as e:
    print("Autoreload not available:", e)

if "google.colab" in sys.modules:
    import getpass
    print("Google Colab detected. Setting up repository...")
    
    %cd /content
    
    token = getpass.getpass("Enter GitHub Token: ")
    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git reset --hard HEAD 
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    !pip install -e . --no-deps
    
else:
    print("Local environment detected. Skipping Git clone/pull.")

Autoreload enabled.
Local environment detected. Skipping Git clone/pull.


In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

2.11.0+cu130
cpu


## **Data Loading & Preprocessing**

In [3]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [4]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [7]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
from torch.amp import GradScaler
import torch.nn as nn

In [9]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.amp.GradScaler(DEVICE)
PATIENCE = 7

## **Training**

In [10]:
from src.trainer import train_model

In [ ]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    scaler=SCALER,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)


Starting training for 100 epochs...



## **Inference**

In [ ]:
from src.evaluation.evaluate_model import evaluate_model

ModuleNotFoundError: No module named 'src.evaluation.evaluate_model'

In [ ]:
eval_results = evaluate_model(
    model=model,
    test_loader=test_loader,
    grapheme_vocab=grapheme_vocab,
    phoneme_vocab=phoneme_vocab,
    device=DEVICE,
)